In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
file_path = "../data/west_bengal_climate_final.nc"

ds = xr.open_dataset(file_path)

print(ds)

<xarray.Dataset> Size: 2MB
Dimensions:             (time: 4018, latitude: 7, longitude: 5)
Coordinates:
  * time                (time) datetime64[ns] 32kB 2015-01-01 ... 2025-12-31
  * latitude            (latitude) float64 56B 21.5 22.5 23.5 ... 25.5 26.5 27.5
  * longitude           (longitude) float64 40B 85.5 86.5 87.5 88.5 89.5
Data variables:
    rainfall            (time, latitude, longitude) float64 1MB ...
    tmax                (time, latitude, longitude) float32 563kB ...
    tmin                (time, latitude, longitude) float32 563kB ...
    overlap_percentage  (latitude, longitude) float64 280B ...
    west_bengal_mask    (latitude, longitude) bool 35B ...
Attributes:
    title:                      VARUNA Climate Dataset
    description:                Daily IMD rainfall, maximum temperature, and ...
    region:                     West Bengal
    rainfall_source:            IMD 0.25 degree gridded rainfall
    temperature_source:         IMD 1 degree gridded temperatu

In [3]:
print("Start date:", ds.time.values[0])
print("End date:", ds.time.values[-1])
print("Number of time steps:", len(ds.time))

Start date: 2015-01-01T00:00:00.000000000
End date: 2025-12-31T00:00:00.000000000
Number of time steps: 4018


In [4]:
is_sorted = np.all(np.diff(ds.time.values.astype('datetime64[D]')) > np.timedelta64(0, 'D'))

print("Time is strictly chronological:", is_sorted)

Time is strictly chronological: True


In [5]:
train = ds.sel(time=slice("2015-01-01", "2022-12-31"))

val = ds.sel(time=slice("2023-01-01", "2024-12-31"))

test = ds.sel(time=slice("2025-01-01", "2025-12-31"))

print("Training:", train.time.values[0], "to", train.time.values[-1])
print("Validation:", val.time.values[0], "to", val.time.values[-1])
print("Test:", test.time.values[0], "to", test.time.values[-1])

Training: 2015-01-01T00:00:00.000000000 to 2022-12-31T00:00:00.000000000
Validation: 2023-01-01T00:00:00.000000000 to 2024-12-31T00:00:00.000000000
Test: 2025-01-01T00:00:00.000000000 to 2025-12-31T00:00:00.000000000


In [6]:
print("Training samples:", len(train.time))
print("Validation samples:", len(val.time))
print("Test samples:", len(test.time))

print("Total:", len(train.time) + len(val.time) + len(test.time))

Training samples: 2922
Validation samples: 731
Test samples: 365
Total: 4018


In [7]:
train_dates = set(train.time.values)
val_dates = set(val.time.values)
test_dates = set(test.time.values)

print("Train ∩ Validation:", len(train_dates & val_dates))
print("Train ∩ Test:", len(train_dates & test_dates))
print("Validation ∩ Test:", len(val_dates & test_dates))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [8]:
variables = ["rainfall", "tmax", "tmin"]

train_mean = {}
train_std = {}

for var in variables:

    # Select only valid West Bengal cells
    valid_train_data = train[var].where(train["west_bengal_mask"])

    train_mean[var] = float(valid_train_data.mean(skipna=True).values)
    train_std[var] = float(valid_train_data.std(skipna=True).values)

print("Training normalization statistics:\n")

for var in variables:
    print(f"{var}:")
    print(f"  Mean = {train_mean[var]:.6f}")
    print(f"  Std  = {train_std[var]:.6f}")

Training normalization statistics:

rainfall:
  Mean = 4.645871
  Std  = 13.518388
tmax:
  Mean = 30.690184
  Std  = 4.541145
tmin:
  Mean = 20.315073
  Std  = 5.800562


In [9]:
for var in variables:
    print(
        var,
        "mean =", train_mean[var],
        "std =", train_std[var],
        "valid std =", train_std[var] > 0
    )

rainfall mean = 4.64587078944167 std = 13.518388116920997 valid std = True
tmax mean = 30.690183639526367 std = 4.541145324707031 valid std = True
tmin mean = 20.315073013305664 std = 5.800561904907227 valid std = True


In [10]:
train_norm = train.copy()
val_norm = val.copy()
test_norm = test.copy()

for var in variables:

    mean = train_mean[var]
    std = train_std[var]

    train_norm[var] = (train[var] - mean) / std
    val_norm[var] = (val[var] - mean) / std
    test_norm[var] = (test[var] - mean) / std

print("Normalization completed.")

Normalization completed.


In [11]:
print("TRAINING NORMALIZATION CHECK\n")

for var in variables:

    valid_data = train_norm[var].where(train_norm["west_bengal_mask"])

    mean_value = float(valid_data.mean(skipna=True).values)
    std_value = float(valid_data.std(skipna=True).values)

    print(f"{var}:")
    print(f"  Mean = {mean_value:.6f}")
    print(f"  Std  = {std_value:.6f}")

TRAINING NORMALIZATION CHECK

rainfall:
  Mean = -0.000000
  Std  = 1.000000
tmax:
  Mean = -0.000001
  Std  = 1.000000
tmin:
  Mean = 0.000000
  Std  = 1.000000


In [12]:
print("VALIDATION AND TEST NORMALIZATION CHECK\n")

for dataset_name, dataset in [
    ("Validation", val_norm),
    ("Test", test_norm)
]:

    print(f"\n{dataset_name}:")

    for var in variables:

        valid_data = dataset[var].where(dataset["west_bengal_mask"])

        mean_value = float(valid_data.mean(skipna=True).values)
        std_value = float(valid_data.std(skipna=True).values)

        print(
            f"  {var}: mean={mean_value:.4f}, std={std_value:.4f}"
        )

VALIDATION AND TEST NORMALIZATION CHECK


Validation:
  rainfall: mean=-0.0114, std=0.9683
  tmax: mean=0.0709, std=1.0780
  tmin: mean=0.1507, std=0.9774

Test:
  rainfall: mean=0.0068, std=0.8729
  tmax: mean=-0.0540, std=0.9731
  tmin: mean=0.1127, std=0.9504


In [13]:
mask = ds["west_bengal_mask"]

print("Mask dimensions:", mask.dims)
print("Mask shape:", mask.shape)
print("Number of valid cells:", int(mask.sum().values))
print("Total grid cells:", mask.size)

Mask dimensions: ('latitude', 'longitude')
Mask shape: (7, 5)
Number of valid cells: 14
Total grid cells: 35


In [14]:
normalization_stats = pd.DataFrame({
    "variable": variables,
    "mean": [train_mean[v] for v in variables],
    "std": [train_std[v] for v in variables]
})

normalization_stats

,variable,mean,std
0,rainfall,4.645871,13.518388
1,tmax,30.690184,4.541145
2,tmin,20.315073,5.800562


In [15]:
stats_path = "../data/normalization_stats.csv"

normalization_stats.to_csv(stats_path, index=False)

print("Saved normalization statistics to:")
print(stats_path)

Saved normalization statistics to:
../data/normalization_stats.csv


In [17]:
processed_dir = "../data/processed"

os.makedirs(processed_dir, exist_ok=True)

train_path = os.path.join(processed_dir, "train_normalized.nc")
val_path = os.path.join(processed_dir, "validation_normalized.nc")
test_path = os.path.join(processed_dir, "test_normalized.nc")

train_norm.to_netcdf(train_path)
val_norm.to_netcdf(val_path)
test_norm.to_netcdf(test_path)

print("Saved:")
print(train_path)
print(val_path)
print(test_path)

Saved:
../data/processed\train_normalized.nc
../data/processed\validation_normalized.nc
../data/processed\test_normalized.nc


In [18]:
train_check = xr.open_dataset(train_path)
val_check = xr.open_dataset(val_path)
test_check = xr.open_dataset(test_path)

print("TRAIN:")
print(train_check)

print("\nVALIDATION:")
print(val_check)

print("\nTEST:")
print(test_check)

TRAIN:
<xarray.Dataset> Size: 2MB
Dimensions:             (time: 2922, latitude: 7, longitude: 5)
Coordinates:
  * time                (time) datetime64[ns] 23kB 2015-01-01 ... 2022-12-31
  * latitude            (latitude) float64 56B 21.5 22.5 23.5 ... 25.5 26.5 27.5
  * longitude           (longitude) float64 40B 85.5 86.5 87.5 88.5 89.5
Data variables:
    rainfall            (time, latitude, longitude) float64 818kB ...
    tmax                (time, latitude, longitude) float32 409kB ...
    tmin                (time, latitude, longitude) float32 409kB ...
    overlap_percentage  (latitude, longitude) float64 280B ...
    west_bengal_mask    (latitude, longitude) bool 35B ...
Attributes:
    title:                      VARUNA Climate Dataset
    description:                Daily IMD rainfall, maximum temperature, and ...
    region:                     West Bengal
    rainfall_source:            IMD 0.25 degree gridded rainfall
    temperature_source:         IMD 1 degree gridded 